# Text Representation

In [134]:
# Load data generated in Session 1 or the provided data splits (see Absalon, W7 Lab)
import pandas as pd

df_train = pd.read_pickle("train_dataframe.pkl")
df_test = pd.read_pickle("test_dataframe.pkl")

# In this session, we will also need to load the metadata file (see Absalon, W9 Lab)
meta_file = 'meta_All_Beauty.json'

# Exercise 1 {-}

Load the [metadata file](https://absalon.ku.dk/courses/80396/files/9386857?module_item_id=2657111) from Absalon and discard any item that was not rated by our subset of users (not in training or test sets). You can refer to the [original metadata file](https://nijianmo.github.io/amazon/index.html) if you want to look up more explanations about the columns of the metada file. Apply preprocessing in this order: lowercasing, tokenizing, stemming, and stopwords removal (including punctuation) to clean up the text from the `title`. Report the vocabulary size before and after the preprocessing. You may have to specify the language for these steps.

In [135]:
import os
import sys
sys.path.append('../')
import pickle
import pandas as pd

# Load the metadata (items)
meta = pd.read_json(meta_file, lines=True)

meta = meta[meta.asin.isin(df_train.asin) | meta.asin.isin(df_test.asin)]

# Sort by a time-related column
meta = meta.sort_values(by='date')

# Drop duplicates, keeping the "latest" based on the sorted order
meta = meta.drop_duplicates(subset='asin', keep='last')
print("Total number of items: ", len(meta))

Total number of items:  84


In [136]:
#Uncomment and run this to install nltk
!pip install nltk

In [137]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer

import string

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
punctuations = set(string.punctuation)

def preprocess(text):
    # lowercasing
    text = text.lower()
    # tokenizing
    tokens = word_tokenize(text)
    # stemming and removing stopwords and punctuation
    tokens = [ps.stem(token) for token in tokens if token not in stop_words and token not in punctuations]
    return tokens

# Compute vocabulary before preprocessing (using lowercased tokens)
vocab_before = set()
for title in meta['title']:
    tokens = word_tokenize(title)
    vocab_before.update(tokens)

# Apply preprocessing and store result in a new column
title_clean_list = meta['title'].apply(preprocess)

# Compute vocabulary after preprocessing
vocab_after = set()
for tokens in title_clean_list:
    vocab_after.update(tokens)

meta['title_clean_str'] = title_clean_list.apply(
    lambda tokens: TreebankWordDetokenizer().detokenize(tokens)
)

print("Vocabulary size before preprocessing:", len(vocab_before))
print("Vocabulary size after preprocessing:", len(vocab_after))

Vocabulary size before preprocessing: 545
Vocabulary size after preprocessing: 471


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Exercise 2

Representation in vector spaces.

## 2.1

Represent all the items from Exercise 1 in a TF-IDF space. Interpret the meaning of the TF-IDF matrix dimensions. Be careful with multiple instances of preprocessing in the process, as default settings for creating the TF-IDF space may include some.

Tip: You may use the library [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) 

In [138]:
# Reset the meta index to ensure row positions match the tfidf_matrix rows
meta = meta.reset_index(drop=True)
from sklearn.feature_extraction.text import TfidfVectorizer

# Represent items in a TF-IDF space without re-preprocessing the pre-detokenized titles
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split(), preprocessor=lambda x: x)
tfidf_matrix = vectorizer.fit_transform(meta['title_clean_str'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# Explanation:
#   - Rows: Each product/item from the metadata.
#   - Columns: Unique terms from the preprocessed title vocabulary.

# Save results for future use

TF-IDF matrix shape: (84, 471)


c:\Users\david\anaconda3\envs\WRS\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


## 2.2

Using the TF-IDF representation, compute the cosine similarity between products with asin `B000FI4S1E`, `B000LIBUBY` and `B000W0C07Y`. Take a look at their features to see whether results make sense with their characteristics. Round your final answer to 3 decimal places.

In [139]:
from sklearn.metrics.pairwise import cosine_similarity

asin1 = 'B000FI4S1E' 
asin2 = 'B000LIBUBY'
asin3 ='B000W0C07Y'

# Find indices of each asin in the meta DataFrame
idx1 = meta.index[meta['asin'] == asin1][0]
idx2 = meta.index[meta['asin'] == asin2][0]
idx3 = meta.index[meta['asin'] == asin3][0]

# Extract corresponding TF-IDF vectors
vec1 = tfidf_matrix[idx1]
vec2 = tfidf_matrix[idx2]
vec3 = tfidf_matrix[idx3]

# Compute cosine similarities individually
sim_1_2 = cosine_similarity(vec1, vec2)[0][0]
sim_1_3 = cosine_similarity(vec1, vec3)[0][0]
sim_2_3 = cosine_similarity(vec2, vec3)[0][0]

# Print results with rounding to 3 decimals
print(f"Similarity between '{asin1}' and '{asin2}': {sim_1_2:.3f}")
print(f"Similarity between '{asin1}' and '{asin3}': {sim_1_3:.3f}")
print(f"Similarity between '{asin2}' and '{asin3}': {sim_2_3:.3f}")

Similarity between 'B000FI4S1E' and 'B000LIBUBY': 0.031
Similarity between 'B000FI4S1E' and 'B000W0C07Y': 0.024
Similarity between 'B000LIBUBY' and 'B000W0C07Y': 0.501


# Exercise 3

Representation in vector spaces with contextual Word Embeddings.

## 3.1.

Represent all the products from Exercise 1 in a vector space using embeddings from a pre-trained BERT model. The final embedding of a product should be the average of the word embeddings from all the words in the 'title'. Critically evaluate this procedure.

What is the vocabulary size of the model? What are the dimensions of the last hidden state?

Tip: you may install the transformers library and use their pretrained [BERT model uncased](https://huggingface.co/bert-base-uncased).

In [140]:
#Uncomment and run the following line to install the transformers library
!pip install transformers

In [141]:
# LOAD TRANSFORMER
"""
If you plan on using a pretrained model, it’s important to use the associated 
pretrained tokenizer: it will split the text you give it in tokens the same way
for the pretraining corpus, and it will use the same correspondence
token to index (that we usually call a vocab) as during pretraining.
"""

# % pip install transformers
import torch
import transformers
assert transformers.__version__ > '4.0.0'

from transformers import BertModel, BertTokenizerFast

# set-up environment
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


modelname = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(modelname)
model = BertModel.from_pretrained(modelname).to(DEVICE)

# Print out the vocabulary size
print(f"Vocabulary size of {len(tokenizer)}. Input dimension: {model.config.hidden_size}.")

Using device: cpu
Vocabulary size of 30522. Input dimension: 768.


In [142]:
# Represent products in a vector space
"""
When using pre-trained models, it is always advised to feed it data similar to what it was trained with. 
Basically, it doesn't hurt to keep all the words in.
However, the effect (or the lack of it) will vary based on corpus and task. 
Decision here: keep them all since pretraining was done that way.
"""

# Function to batch encode and compute embeddings
def batch_encoding(sentences):
    # Tokenize input sentences with padding and attention mask
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    
    # Forward pass through the model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract last hidden states
    last_hidden_states = outputs.last_hidden_state  # Shape: (batch_size, seq_len, hidden_dim)
    
    return inputs, last_hidden_states

# Encode product titles
encoded_inputs, title_last_hidden_states = batch_encoding(meta["title"].tolist())

# Compute average embeddings by taking the mean across token dimension
meta["bert_embedding"] = [emb.mean(dim=0).cpu().numpy() for emb in title_last_hidden_states]

# Print shape of last hidden states
print(f"Last hidden states shape: {title_last_hidden_states.shape}")

Last hidden states shape: torch.Size([84, 52, 768])


## 3.2.

Using the representation obtained from Exercise 3.1., compute the cosine similarity between items with asin `B000FI4S1E`, `B000LIBUBY` and `B000W0C07Y`.
Round your final answer to 3 decimal places.

In [143]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ASINs to compare
asin1 = 'B000FI4S1E'
asin2 = 'B000LIBUBY'
asin3 = 'B000W0C07Y'

# Find indices of each ASIN in the meta DataFrame
idx1 = meta.index[meta['asin'] == asin1][0]
idx2 = meta.index[meta['asin'] == asin2][0]
idx3 = meta.index[meta['asin'] == asin3][0]

# Extract corresponding BERT embeddings
vec1 = np.array(meta['bert_embedding'][idx1]).reshape(1, -1)
vec2 = np.array(meta['bert_embedding'][idx2]).reshape(1, -1)
vec3 = np.array(meta['bert_embedding'][idx3]).reshape(1, -1)

# Compute cosine similarities individually
sim_1_2 = cosine_similarity(vec1, vec2)[0][0]
sim_1_3 = cosine_similarity(vec1, vec3)[0][0]
sim_2_3 = cosine_similarity(vec2, vec3)[0][0]

# Print results with rounding to 3 decimals
print(f"Similarity between '{asin1}' and '{asin2}': {sim_1_2:.3f}")
print(f"Similarity between '{asin1}' and '{asin3}': {sim_1_3:.3f}")
print(f"Similarity between '{asin2}' and '{asin3}': {sim_2_3:.3f}")


Similarity between 'B000FI4S1E' and 'B000LIBUBY': 0.836
Similarity between 'B000FI4S1E' and 'B000W0C07Y': 0.759
Similarity between 'B000LIBUBY' and 'B000W0C07Y': 0.754
